In [ ]:

# https://github.com/gisbi-kim/PyICP-SLAM/blob/master/main_icp_slam.py

# pip3 install rosbags ros2-numpy opencv-python scikit-learn numpy gtsam open3d
# sudo apt install python3-matplotlib

%load_ext autoreload
%autoreload 2

import os
import sys

from dataengine.source import DataSources
from dataengine.buffer import DataBuffer

from rosbags.highlevel import AnyReader
from rosbags.serde import deserialize_cdr

import numpy as np
from pathlib import Path

from dataengine.visualizers.point_cloud import VisTool

import open3d as o3d
import open3d.visualization.gui as gui
import open3d.visualization.rendering as rendering

from dataengine.point_cloud.point_cloud import o3d_icp




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:

## ROS2 LOAM data
file_path = os.environ['HOME'] + "/repos/monorepo/external_files/data/LOAM_ROS_2/2018-05-18-14-49-12_0/2018-05-18-14-49-12_0.db3"
data = DataSources(file_path)


In [32]:

## Buffer class

# depth of buffer as message count 
buffer_depth = 50
buffer = DataBuffer(data, buffer_depth)

print(buffer.get_topics())


['/imu/data', '/velodyne_points', '/tf']


In [33]:

## Axis

print(list(buffer.get_topics()))

axis = "/velodyne_points"


['/imu/data', '/velodyne_points', '/tf']


In [34]:

## PointCloud

buffer.roll_buffer(axis)
data = buffer[-1]

print("shape: ", data.shape)

vis_tool = VisTool()
# vis_tool = VisTool(embed=False)

vis_tool.show_point_cloud(data)


shape:  (30000, 3)
[Open3D INFO] Window window_10 created.


WebVisualizer(window_uid='window_10')

In [35]:


## LOAM


# https://www.open3d.org/docs/release/python_example/visualization/index.html

# Roll buffer
buffer.roll_buffer(axis)

vis_tool = VisTool(embed=False)

num_icp_points = 1000
odom_transform = np.eye(4)
icp_initial = np.eye(4)
curr_se3 = np.eye(4)

idx = 0

stop = 30

print("axis: ", axis)

# iterator = DataIterator(buffer, axis)
for buf, count in buffer.get_data():
    prev_message = buf[axis]['data'][-2]
    last_message = buf[axis]['data'][-1]

    ## ICP
    odom_transform = o3d_icp(last_message, prev_message, icp_initial)

    ## update the current (moved) pose 
    curr_se3 = np.matmul(curr_se3, odom_transform)

    ## Pose
    vis_tool.add_pose_arrow(curr_se3)
    
    ## Point Cloud
    vis_tool.update_point_cloud(last_message, odom_transform)

    idx += 1
    if idx > stop:
        break

vis_tool.destroy()

print("done")


axis:  /velodyne_points
done


[1824217:382][3831588] (stun_port.cc:96): Binding request timed out from 10.100.33.x:40929 (enx605b305cb72a)
[1824217:382][3831588] (stun_port.cc:96): Binding request timed out from 10.100.143.x:50849 (wlo1)
[1824217:383][3831588] (stun_port.cc:96): Binding request timed out from 100.64.0.x:59512 (zcctun0)
[1824217:407][3831588] (stun_port.cc:96): Binding request timed out from 100.64.0.x:59512 (zcctun0)
[1824217:407][3831588] (stun_port.cc:96): Binding request timed out from 10.100.143.x:50849 (wlo1)


In [ ]:


# TODO: LOAM, loop closure

